# SQL Practice Tasks - E-commerce Store

Welcome! This assignment uses a small **e-commerce** SQLite database
(`sqlite.db`) with ~120,000 rows. Your job is to write the SQL query for each
task in the code cell below its description.

## How to work on this

1. **Fork** this repository and clone your fork.
2. Create and activate a virtual environment:
   - Windows (PowerShell): `python -m venv .venv` then `.venv\Scripts\Activate.ps1`
   - macOS / Linux (bash): `python3 -m venv .venv` then `source .venv/bin/activate`
3. With the environment active, install the requirements: `pip install -r requirements.txt`.
4. Open this notebook and, for each task, write your SQL inside the
   `task_NN = """ ... """` string. Run the cell to see your result.
5. Commit your changes and open a **Pull Request**.

Do **not** edit `sqlite.db`, the helper cell, or the task
descriptions - only fill in the `task_NN` query strings.

## The database

| Table | Description | Key relations |
|-------|-------------|---------------|
| `suppliers` | Companies that supply products | - |
| `categories` | Product categories, **tree** via `parent_id` | self -> `categories` |
| `products` | Products for sale | -> `categories`, `suppliers` |
| `customers` | Registered customers | - |
| `addresses` | Customer addresses (many per customer) | -> `customers` |
| `orders` | Orders placed by customers | -> `customers` |
| `order_items` | Line items (links orders <-> products) | -> `orders`, `products` |
| `reviews` | Product reviews by customers | -> `products`, `customers` |
| `payments` | Payment for a completed order | -> `orders` |

`order_date`/`created_at`/`paid_at` are stored as ISO text (`YYYY-MM-DD`).
Order `status` is one of: `pending`, `paid`, `shipped`, `delivered`, `cancelled`.

Run the **setup cell** below once, then start with Task 1.


In [2]:
import os
import shutil
import sqlite3

import pandas as pd

DB = "sqlite.db"


def _has_statement(sql):
    """True if sql contains anything other than blank lines / -- comments."""
    for line in sql.splitlines():
        s = line.strip()
        if s and not s.startswith("--"):
            return True
    return False


def run_query(sql, db=DB):
    if not _has_statement(sql):
        return pd.DataFrame()
    with sqlite3.connect(db) as conn:
        return pd.read_sql_query(sql, conn)


def run_dml(sql, verify=None, db=DB):
    if not _has_statement(sql):
        return pd.DataFrame()
    copy = "_scratch.db"
    shutil.copyfile(db, copy)
    try:
        conn = sqlite3.connect(copy)
        conn.execute("PRAGMA foreign_keys = ON")
        conn.executescript(sql)
        conn.commit()
        result = pd.read_sql_query(verify, conn) if verify else None
        conn.close()
        return result
    finally:
        if os.path.exists(copy):
            os.remove(copy)


---
## Part A - Data Query Language (DQL)


### Task 1: Most expensive products

Return the `product_id`, `name` and `price` of the **10 most expensive** products, most expensive first.

In [4]:
task_01 = """
SELECT product_id, name, price FROM products
ORDER BY price DESC
LIMIT 10
"""
run_query(task_01)

,product_id,name,price
0,252,Essential Sneakers 1252,1499.97
1,1531,Turbo Lamp 2531,1498.24
2,1111,Turbo Drone 2111,1496.19
3,854,Compact Widget 1854,1495.24
4,769,Max Chair 1769,1494.96
5,1724,Essential Speaker 2724,1494.47
6,1332,Eco Keyboard 2332,1494.35
7,690,Classic Router 1690,1494.22
8,999,Smart Desk 1999,1494.00
9,1153,Pro Camera 2153,1493.95


### Task 2: Customers from Germany

Return `first_name`, `last_name` and `email` of all customers whose `country` is `'Germany'`, ordered by `last_name`, then `first_name`.

In [31]:
task_02 = """
SELECT first_name, last_name, email FROM customers
WHERE country = "Germany"
ORDER BY last_name, first_name
"""
run_query(task_02)

,first_name,last_name,email
0,Alex,Andersson,alex.andersson2702@example.com
1,Alex,Andersson,alex.andersson4912@example.com
2,Anna,Andersson,anna.andersson407@example.com
3,Hannah,Andersson,hannah.andersson2118@example.com
4,Hannah,Andersson,hannah.andersson4421@example.com
...,...,...,...
329,Olena,Williams,olena.williams3762@example.com
330,Sofia,Williams,sofia.williams2999@example.com
331,Sofia,Williams,sofia.williams4921@example.com
332,Yuki,Williams,yuki.williams56@example.com


### Task 3: Mid-priced 'Pro' products

Return `name` and `price` of products whose `name` contains `Pro` **and** whose `price` is between 100 and 500 (inclusive). Order by `price`.

In [13]:
task_03 = """
SELECT name, price FROM products
WHERE name LIKE '%Pro%' AND price BETWEEN 100 AND 500
ORDER BY price
"""
run_query(task_03)

,name,price
0,Pro Desk 1245,101.67
1,Pro Charger 2500,104.14
2,Pro Jacket 1021,105.58
3,Pro Headset 1001,131.22
4,Pro Charger 2083,131.92
5,Pro Desk 2322,148.73
6,Pro Drone 2145,158.55
7,Pro Router 2301,160.44
8,Pro Notebook 2107,167.22
9,Pro Speaker 2454,173.61


### Task 4: Orders per status

For each order `status`, return the `status` and the number of orders as `order_count`.

In [16]:
task_04 = """
SELECT status, COUNT(*) AS order_count FROM orders
GROUP BY status
"""
run_query(task_04)

,status,order_count
0,cancelled,1377
1,delivered,4894
2,paid,2831
3,pending,1461
4,shipped,3437


### Task 5: Countries with many customers

Return each `country` and its number of customers as `customer_count`, but only for countries with **more than 340 customers**. Order by `customer_count` descending.

In [32]:
task_05 = """
SELECT country, COUNT(*) AS customer_count FROM customers
GROUP BY country
HAVING customer_count > 340
ORDER BY customer_count DESC
"""
run_query(task_05)

,country,customer_count
0,Brazil,371
1,UK,350
2,Netherlands,345
3,Spain,342


### Task 6: Products with their category

Return the product `name` (as `product_name`) and its category `name` (as `category`) for the first 20 products by `product_id`.

In [25]:
task_06 = """
SELECT p.name AS product_name, c.name AS category_name
FROM products p
JOIN categories c ON p.category_id = c.category_id
LIMIT 20
"""
run_query(task_06)

,product_name,category_name
0,Pro Headset 1001,Men
1,Essential Notebook 1002,Fragrance
2,Mini Keyboard 1003,Camping
3,Essential Backpack 1004,Outdoor Toys
4,Smart Watch 1005,Camping
5,Smart Widget 1006,Men
6,Ultra Desk 1007,Home & Kitchen
7,Ultra Jacket 1008,Automotive
8,Mini Mug 1009,Men
9,Max Charger 1010,Fashion


### Task 7: Best-selling categories by quantity

For each category, return the category `name` (as `category`) and the **total quantity sold** (`SUM(order_items.quantity)`) as `total_qty`. Show the top 10 categories by `total_qty`.

In [29]:
task_07 = """
SELECT c.name AS category, SUM(oi.quantity) AS total_qty
FROM categories c
JOIN products p ON p.category_id = c.category_id
JOIN order_items oi ON oi.product_id = p.product_id
GROUP BY c.name
ORDER BY total_qty DESC
LIMIT 10
"""
run_query(task_07)

,category,total_qty
0,Men,8735
1,Home & Kitchen,8569
2,Cookware,8358
3,Fitness,8312
4,Computers,8123
5,Skincare,7595
6,Automotive,7486
7,Camping,7452
8,Board Games,7196
9,Fashion,7142


### Task 8: Products never ordered

Return the `product_id` and `name` of products that have **never been ordered** (no rows in `order_items`). Order by `product_id`.

In [33]:
task_08 = """
SELECT p.product_id AS product_id, p.name
FROM products p
LEFT JOIN order_items oi ON oi.product_id = p.product_id
WHERE oi.product_id IS NULL
ORDER BY product_id
"""
run_query(task_08)

,product_id,name
0,1851,Eco Keyboard 2851
1,1852,Max Mouse 2852
2,1853,Nano Lamp 2853
3,1854,Lite Charger 2854
4,1855,Smart Speaker 2855
...,...,...
145,1996,Turbo Keyboard 2996
146,1997,Eco Bottle 2997
147,1998,Nano Watch 2998
148,1999,Plus Charger 2999


### Task 9: Products above the average price

Return `name` and `price` of products priced **above the overall average product price**. Use a subquery. Order by `price` descending.

In [37]:
task_09 = """
SELECT name, price FROM products
WHERE price > (SELECT AVG(price) FROM products)
ORDER BY price DESC
"""
run_query(task_09)

,name,price
0,Essential Sneakers 1252,1499.97
1,Turbo Lamp 2531,1498.24
2,Turbo Drone 2111,1496.19
3,Compact Widget 1854,1495.24
4,Max Chair 1769,1494.96
...,...,...
1001,Pro Charger 1472,767.48
1002,Eco Blender 1891,767.14
1003,Smart Drone 2829,766.27
1004,Compact Charger 1553,764.71


### Task 10: Products above their category average

Return `name`, `category_id` and `price` of products whose price is above the **average price of their own category**. Order by `category_id`, then `price` descending.

In [38]:
task_10 = """
SELECT p1.name, p1.category_id, p1.price FROM products p1
WHERE p1.price > (
    SELECT AVG(p2.price)
    FROM products p2
    WHERE p2.category_id = p1.category_id
)
ORDER BY price DESC
"""
run_query(task_10)

,name,category_id,price
0,Essential Sneakers 1252,7,1499.97
1,Turbo Lamp 2531,24,1498.24
2,Turbo Drone 2111,5,1496.19
3,Compact Widget 1854,6,1495.24
4,Max Chair 1769,17,1494.96
...,...,...,...
1012,Turbo Widget 1133,22,694.58
1013,Smart Mug 2385,22,688.87
1014,Eco Camera 1080,11,683.68
1015,Lite Chair 2603,11,678.37


### Task 11: Subcategories and their parents

Each subcategory has a `parent_id` pointing at another category. Return each subcategory `name` (as `child_name`) together with its parent's `name` (as `parent_name`). Order by `parent_name`, then `child_name`.

In [8]:
task_11 = """
SELECT c1.name AS child_name, c2.name AS parent_name FROM categories c1
JOIN categories c2 ON c1.category_id = c2.parent_id
ORDER BY parent_name, child_name

"""
run_query(task_11)

,child_name,parent_name
0,Home & Kitchen,Appliances
1,Electronics,Audio
2,Toys & Games,Board Games
3,Electronics,Cameras
4,Sports & Outdoors,Camping
5,Automotive,Car Electronics
6,Electronics,Computers
7,Home & Kitchen,Cookware
8,Sports & Outdoors,Cycling
9,Sports & Outdoors,Fitness


### Task 12: Expensive and cheap products labelled

Build a single result that labels products: those with `price > 1000` get the label `'expensive'`, those with `price < 10` get `'cheap'`. Return `name`, `price`, `label`. Order by `label`, then `price`.

In [11]:
task_12 = """
SELECT name, price,
    CASE
        WHEN price > 1000 THEN "expensive"
        WHEN price < 10 THEN "cheap"
    END AS label
FROM products
WHERE price > 1000 OR price < 10
ORDER BY label, price
"""
run_query(task_12)

,name,price,label
0,Compact Watch 2216,5.04,cheap
1,Nano Gadget 1016,6.23,cheap
2,Premium Sneakers 1512,6.70,cheap
3,Smart Lamp 1189,7.24,cheap
4,Smart Monitor 2715,7.62,cheap
...,...,...,...
686,Max Chair 1769,1494.96,expensive
687,Compact Widget 1854,1495.24,expensive
688,Turbo Drone 2111,1496.19,expensive
689,Turbo Lamp 2531,1498.24,expensive


### Task 13: Top customers by total spend

Return the top 10 customers by total amount spent. For each, return `customer_id`, `first_name`, `last_name`, `country` and `total_spent` (the sum of `quantity * unit_price` over all their order items, rounded to 2 decimals). Order by `total_spent` descending.

In [21]:
task_13 = """
SELECT c.customer_id, c.first_name, c.last_name, c.country, ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_spent
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY c.customer_id
ORDER BY total_spent DESC
"""
run_query(task_13)

,customer_id,first_name,last_name,country,total_spent
0,962,Pedro,Tanaka,Netherlands,112948.26
1,3363,David,Rossi,Spain,112918.70
2,4978,Alex,Tanaka,Norway,111177.87
3,2682,Marco,Garcia,Italy,106141.79
4,2845,Marco,Ferrari,Canada,106061.84
...,...,...,...,...,...
4675,1923,Yuki,Shevchenko,France,231.55
4676,1386,Michael,Ferrari,Brazil,169.40
4677,2197,Daniel,Martin,Canada,141.64
4678,3627,Ivan,Jansen,Japan,90.28


### Task 14: Line items of a single order

List every line item of order `order_id = 1`. Return the `order_id`, the customer's `last_name` (as `customer`), the product `name` (as `product`), the `quantity` and the `unit_price`. Order by `product`.

In [24]:
task_14 = """
SELECT o.order_id, c.last_name AS customer, p.name AS product, oi.quantity, oi.unit_price
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN customers c ON o.customer_id = c.customer_id
join products p ON oi.product_id = p.product_id
WHERE o.order_id = 1
ORDER BY product
"""
run_query(task_14)

,order_id,customer,product,quantity,unit_price
0,1,Schmidt,Classic Backpack 2600,1,1333.40
1,1,Schmidt,Classic Charger 2784,4,266.07
2,1,Schmidt,Classic Headset 2016,3,1482.40
3,1,Schmidt,Compact Backpack 2077,5,291.24
4,1,Schmidt,Deluxe Lamp 1525,1,345.73
5,1,Schmidt,Mini Bottle 1957,4,758.53
6,1,Schmidt,Mini Sneakers 1286,5,552.92
7,1,Schmidt,Plus Keyboard 2395,3,655.36


### Task 15: Customers who never ordered

Return the `customer_id`, `first_name` and `last_name` of every customer who has **never placed an order**. Use a subquery against the `orders` table. Order by `customer_id`.

In [29]:
task_15 = """
SELECT c.customer_id, c.first_name, c.last_name
FROM customers c
WHERE c.customer_id NOT IN (
    SELECT customer_id FROM orders
)
ORDER BY customer_id
"""
run_query(task_15)

,customer_id,first_name,last_name
0,5,Robert,Tanaka
1,10,Robert,Rossi
2,14,Olena,Bernard
3,70,Emma,Silva
4,78,Alex,Williams
...,...,...,...
315,4893,John,Walker
316,4902,Greta,Martin
317,4917,James,Dubois
318,4940,Olena,Shevchenko


---
## Part B - Data Manipulation Language (DML)

Tasks 16-21 change data. They run with `run_dml(...)`, which executes your
statements against a **temporary copy** of the database - the original
`sqlite.db` is never touched, so you can run them as often as you like. The
pre-filled `verify=...` SELECT shows the effect of your change.


### Task 16: Add a new supplier

Insert a new supplier with `supplier_id = 999`, `name = 'Test Supplier'`, `country = 'USA'`, `email = 'test@supplier.example.com'`, `rating = 4.5`.

In [35]:
task_16 = """
INSERT INTO suppliers (supplier_id, name, country, email, rating)
VALUES (999, 'Test Supplier', 'USA', 'test@supplier.example.com', 4.5)
"""
run_dml(task_16, verify="""SELECT * FROM suppliers WHERE supplier_id = 999;""")

,supplier_id,name,country,email,rating
0,999,Test Supplier,USA,test@supplier.example.com,4.5


### Task 17: Add two subcategories

Insert two new subcategories under `parent_id = 1` (Electronics) in a single statement: `category_id = 100, name = 'Wearables'` and `category_id = 101, name = 'Smart Home'`.

In [39]:
task_17 = """
INSERT INTO categories (category_id, name, parent_id)
VALUES
    (100, 'Wearables', 1),
    (101, 'Smart Home', 1)
"""
run_dml(task_17, verify="""SELECT * FROM categories WHERE category_id IN (100, 101) ORDER BY category_id;""")

,category_id,name,parent_id
0,100,Wearables,1
1,101,Smart Home,1


### Task 18: Raise prices in a category

Increase the `price` of every product in `category_id = 1` by 10% (round the result to 2 decimals).

In [44]:
task_18 = """
UPDATE products
SET price = ROUND(price * 1.1, 2)
WHERE category_id = 1
"""
run_dml(task_18, verify="""SELECT ROUND(AVG(price), 2) AS avg_price FROM products WHERE category_id = 1;""")

,avg_price
0,944.53


### Task 19: Clear stock for unsold products

Set `stock = 0` for every product that has **never been ordered** (no matching row in `order_items`). Use a subquery.

In [46]:
task_19 = """
UPDATE products
SET stock = 0
WHERE product_id NOT IN (
    SELECT product_id FROM order_items
    )
"""
run_dml(task_19, verify="""SELECT COUNT(*) AS products_with_zero_stock FROM products WHERE stock = 0;""")

,products_with_zero_stock
0,154


### Task 20: Remove one-star reviews

Delete every review whose `rating = 1`.

In [52]:
task_20 = """
DELETE FROM reviews
WHERE rating = 1
"""
run_dml(task_20, verify="""SELECT COUNT(*) AS remaining_one_star FROM reviews WHERE rating = 1;""")

,remaining_one_star
0,0


### Task 21: Create an order with line items

Inside a single transaction (`BEGIN; ... COMMIT;`), create a new order with `order_id = 100000` for `customer_id = 1`, `order_date = '2026-06-01'`, `status = 'pending'`; then add two `order_items` for products 1 and 2 (quantity 1 each, `unit_price` = each product's current `price`). Let the `order_item_id` auto-generate.

In [3]:
task_21 = """
BEGIN;

INSERT INTO orders (order_id, customer_id, order_date, status)
VALUES (100000, 1, '2026-06-01', 'pending');

INSERT INTO order_items (order_id, product_id, quantity, unit_price)
SELECT 100000, product_id, 1, price
FROM products
WHERE product_id in (1, 2);

COMMIT;
"""
run_dml(task_21, verify="""SELECT o.order_id, COUNT(oi.order_item_id) AS items
FROM orders o JOIN order_items oi ON oi.order_id = o.order_id
WHERE o.order_id = 100000
GROUP BY o.order_id;""")

,order_id,items
0,100000,2
